# Phase 1.1: Data toy-load (Munich / Alpine foreland)

One small, trustworthy slice of real data in memory, plus the single canonical
NDVI definition the rest of the project depends on.

## Before you run anything

This notebook does not contain the project code. It imports it, because
`data/ndvi.py` must have exactly one definition in the world. So the repo has to
be on your Drive first:

1. Open Google Drive.
2. Drag `phase1_1_repo.zip` into `My Drive/NeurIPS-CCAI-2026/`.
3. Do not unzip it by hand. Step 2 unzips it in the right place.

To update the code later, drop the new zip in the same place, overwrite, and
re-run Step 2. It re-extracts when the zip is newer.

## Runtime

Runtime > Change runtime type > T4 GPU, before Step 1. Changing it later
restarts the kernel and wastes the install.

A GPU does not speed up this phase. The cloud mask runs on CPU. Set it now so
Phase 1.2 is ready.

## Restarts: exactly one

Step 1 installs, then restarts the kernel on purpose. That is expected, not a
crash. After it comes back, run from Step 2 down. Step 1 self-skips.

## Steps and cost

| step | what | time |
|---|---|---|
| 1 | install, then auto-restart | 3 min |
| 2 | bootstrap: mount, unzip, paths | 1 min |
| 3 | environment check | instant |
| 4 | `ndvi()` unit tests | 15 s |
| 5 | diagnostic | 90 s |
| 6 | download 20 cubes | 15 s |
| 7 to 9 | load, check, plot | 2 min |

Step 6 re-runs cheaply. Finished cubes are skipped.

## Exit criteria

1. `ndvi()` unit test green.
2. Up to 20 minicubes load.
3. Timestamps parse and are irregular.
4. Mask fraction printed per cube.

## Step 1: Install, then restart

Run once. Ends by restarting the kernel.

In [ ]:
import os, IPython
SENTINEL = "/content/.phase1_1_installed"

if os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the most likely reason the
    # download later produces nothing, and -q hides it.
    # s3fs is what actually fetches the cubes. earthnet is the toolkit, kept
    # for its plotting and scoring helpers.
    !pip install earthnet s3fs xarray zarr netCDF4

    # Colab ships a CUDA-matched torch. Installing over it swaps in a CPU wheel
    # and makes every later phase far slower, so only act if it is missing.
    import importlib.util
    if importlib.util.find_spec("torch") is None:
        !pip install torch torchvision

    # Verify before restarting, so a broken install cannot reach step 6.
    import subprocess, sys
    probe = "import s3fs, xarray, zarr, netCDF4, earthnet"
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: the download would fail and leave "
            "data/raw empty."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

## Step 2: Bootstrap

Mounts Drive, unzips the repo, and proves it is importable.

Also sets `PYTHONPATH`. The download runs in a subprocess, and a subprocess does
not inherit `sys.path` edits.

In [ ]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/download_greenearthnet.py",
            "data/diagnose.py", "tests/test_ndvi.py", "tests/conftest.py"]
ZIP_NAME = "phase1_1_repo.zip"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for h in glob.glob(f"{DRIVE}/*/data/ndvi.py")
                + glob.glob(f"{DRIVE}/*/*/data/ndvi.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "data", "ndvi.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    REPO = os.getcwd()

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the project code.

        Fix, 2 minutes:
          1. Open https://drive.google.com
          2. Create the folder:  My Drive / NeurIPS-CCAI-2026
          3. Drag {ZIP_NAME} into it. Do not unzip it yourself.
          4. Re-run this cell.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

RAW = os.path.join(REPO, "data", "raw")     # on Drive, so it survives a disconnect
os.makedirs(RAW, exist_ok=True)

print(f"\nREPO  {REPO}")
print(f"RAW   {RAW}")
for f in REQUIRED:
    print(f"  ok  {f}")

from data.ndvi import ndvi
from data.loader import iter_cubes, describe_cube, cube_ndvi, assert_no_overlap, S2_BANDS
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, bands {S2_BANDS}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
#
# PYTHONUNBUFFERED matters: a subprocess writing to a pipe block-buffers stdout,
# so progress lines sit invisible for minutes then arrive in one burst. That is
# what makes a working download look like a hang.
import subprocess

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

## Step 3: Environment check

In [ ]:
import torch, xarray, zarr, s3fs, numpy as np, shutil

print("torch     ", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY")
print("xarray    ", xarray.__version__)
print("s3fs      ", s3fs.__version__)
print("zarr      ", zarr.__version__)
print("numpy     ", np.__version__)

if torch.cuda.is_available():
    print(f"GPU mem    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
          "  (budget: T4 <=12 GB, chair GPU <=24 GB)")
else:
    print("no GPU. Phase 1.1 needs none, the cubes are pre-processed.")

free = shutil.disk_usage(RAW).free / 1e9
print(f"free space {free:.1f} GB at {RAW}  (need about 0.1 GB for 20 cubes)")
assert free > 1, "less than 1 GB free, clear space on Drive before downloading"

## Step 4: Unit tests

`tests/test_ndvi.py` was written before `data/ndvi.py` existed. The 2x2 toy has
hand-computed NDVI and one cloud-masked pixel whose reflectances would give a
finite -0.5 if the mask were dropped, inverted, or applied after aggregation.

Expect `48 passed`. Anything else: stop, do not download.

In [ ]:
sh("python -m pytest tests -q")

In [ ]:
# The definition itself. Read it once, then never re-implement it.
import inspect
from data.ndvi import ndvi

print(inspect.getsource(ndvi).split(chr(34) * 3)[0] + "...")
print("canonical NDVI lives at:", inspect.getsourcefile(ndvi))

## Step 5: Diagnostic

Four escalating checks: imports, STAC query, cloud-mask checkpoint, and one real
one-month cube. Stops at the first failure with its full traceback.

Run this before reporting any problem, and paste its output verbatim.

In [ ]:
sh("python -m data.diagnose")

## Step 6: Download the cubes

Pre-processed GreenEarthNet minicubes, pulled straight from the MPG object
store. About 3.4 MB each, so 20 cubes is roughly 70 MB and 15 seconds.

Tile `32UNU`, 9.00 to 10.49 E, 47.76 to 48.75 N: Allgaeu and Upper Swabia. This
is the closest Alpine-foreland tile the dataset actually contains. `32UPU`,
which holds Munich itself, is not in GreenEarthNet. Same latitude band as
Munich, about 135 km west, same landscape of grassland, arable and forest.

Selection takes 20 of the tile's 192 cubes, round-robin across the 16 available
time windows for seasonal spread, and requires 64 px of separation between
footprints so no two cubes are adjacent, let alone overlapping.

Each cube is 128 x 128 px at 20 m over a 150-day window on a daily grid, of
which about 29 days carry a Sentinel-2 acquisition. The loader drops the empty
days, which is where the irregular time axis comes from.

In [ ]:
sh(f"python -m data.download_greenearthnet --out '{RAW}' --n 20 --tile 32UNU")

### If you need the live-extraction path instead

`data/download_minicubes.py` still builds cubes from Sentinel-2 via Planetary
Computer, at any location and date range you like. It works, and it measured
14.7 hours for 20 cubes because the cloud-mask U-Net runs on CPU. Use it only
when you need a location GreenEarthNet does not cover.

## Step 7: Load the cubes

Exit criteria 2, 3 and 4: cubes load, timestamps parse, mask fractions printed.

In [ ]:
import numpy as np, pandas as pd
from data.loader import iter_cubes, describe_cube, cube_ndvi, assert_no_overlap, S2_BANDS

rows, samples = [], []
for s in iter_cubes(RAW, limit=20):
    rows.append(describe_cube(s))
    samples.append(s)
    print()

df = pd.DataFrame(rows)
print(f"\n{len(samples)} cubes loaded, band order {S2_BANDS}")
df[["name", "T", "H", "W", "masked_fraction", "n_steps_gt50pct_clear",
    "dt_days_median", "dt_days_max", "refl_min", "refl_max",
    "ndvi_valid_fraction", "ndvi_median"]]

### How to read that table

| column | healthy | a bad value means |
|---|---|---|
| `T` | 25 to 35 | far off: the empty daily slots were not dropped |
| `masked_fraction` | 0.45 to 0.75 | near 0: masking off or polarity inverted. Near 1: inverted the other way |
| `n_steps_gt50pct_clear` | 8 to 20 | under 6 leaves too little signal for Phase 1.2 |
| `dt_days_median` | 2 to 5 | 5.0 with `dt_days_max` also 5.0: the grid got regularised |
| `refl_min`, `refl_max` | about 0, under 2.5 | negative: BOA offset present, NDVI is wrong. Values above 1 are bright cloud tops, which is normal |
| `ndvi_median` | 0.4 to 0.8 | strongly negative: B04 and B8A are swapped |

Bavaria is cloudy. A masked fraction near 0.6 is the correct answer, not a
problem.

The next cell asserts these bounds, plus that no two cubes share pixels.

In [ ]:
import numpy as np
from data.loader import assert_no_overlap

assert "samples" in globals(), "Run the previous cell (Step 8) first."

# Leakage guard: no two cubes on disk may share pixels. A cube left over from an
# earlier run with different settings is the usual cause.
assert_no_overlap(RAW)

assert 0 < len(samples) <= 20, f"expected 1 to 20 cubes, got {len(samples)}"
for s in samples:
    T, C, H, W = s.values.shape
    assert (C, H, W) == (4, 128, 128), f"{s.path}: unexpected shape {s.values.shape}"
    assert 20 <= T <= 40, f"{s.path}: expected about 29 acquisitions, got {T}"
    assert s.mask.shape == (T, H, W) and s.mask.dtype == np.bool_
    assert s.timestamps.shape == (T,) and s.timestamps.dtype == np.dtype("datetime64[ns]")
    assert np.all(np.diff(s.timestamps) > np.timedelta64(0, "ns")), f"{s.path}: time not sorted"

print("masked fraction : min %.3f  median %.3f  max %.3f"
      % (df.masked_fraction.min(), df.masked_fraction.median(), df.masked_fraction.max()))
print("timesteps       : min %d  median %.0f  max %d"
      % (df["T"].min(), df["T"].median(), df["T"].max()))
print("reflectance     : global min %.4f  max %.4f" % (df.refl_min.min(), df.refl_max.max()))
print("NDVI median     : min %.3f  max %.3f" % (df.ndvi_median.min(), df.ndvi_median.max()))

assert df.refl_min.min() > -0.05, "negative reflectance: BOA offset not removed, NDVI is wrong"
assert df.refl_max.max() < 3.0, "reflectance out of range: scaling not harmonised"
assert df.masked_fraction.max() < 0.98, "a cube is almost fully masked, check the cloud mask"
assert df.masked_fraction.min() > 0.02, "a cube is almost never masked, mask polarity inverted"
assert df.ndvi_median.min() > 0.0, "negative median NDVI over Bavaria: B04 and B8A swapped"
print("\nALL SHAPE AND RANGE ASSERTIONS PASSED")

## Step 8: The time grid is irregular

Look at it before trusting any model.

If `exactly 5 d` comes out at 100%, something upstream regularised the grid, and
the forecastability you are about to measure is an artifact of interpolation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

assert "samples" in globals(), "Run Step 7 first."

all_dt = np.concatenate([np.diff(s.timestamps) / np.timedelta64(1, "D") for s in samples])
print("dt/days: n=%d  min=%.0f  p50=%.0f  p90=%.0f  max=%.0f  |  exactly 5 d: %.1f%%"
      % (all_dt.size, all_dt.min(), np.percentile(all_dt, 50),
         np.percentile(all_dt, 90), all_dt.max(), 100 * np.mean(all_dt == 5)))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].hist(all_dt, bins=np.arange(0, all_dt.max() + 5, 2.5))
ax[0].set(xlabel="gap between acquisitions [days]", ylabel="count",
          title="Nominal 5-daily, realised irregular")
for i, s in enumerate(samples[:8]):
    ax[1].plot(s.timestamps, np.full(s.timestamps.size, i), "|", ms=8)
ax[1].set(yticks=range(min(8, len(samples))), ylabel="cube", title="acquisition times")
plt.tight_layout()
plt.show()

## Step 9: NDVI sanity

Every number below came from `data.ndvi.ndvi`.

Each cube covers one 150-day window in 2018, so expect a single green-up, not
three annual humps: NDVI rising from about 0.3 to 0.4 in early spring to 0.7 or
0.8 by midsummer. The map should show fields as a patchwork and forest as
uniformly dark.

Two failure signatures worth knowing by sight:

- A flat line near 0.2 to 0.3 means you are averaging clouds, not vegetation.
- An NDVI curve that mirrors the clear-fraction panel means the mask is
  correlated with the signal. Suspect polarity or a broken checkpoint.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from data.loader import cube_ndvi

assert "samples" in globals(), "Run Step 7 first."

s = samples[0]
nd = cube_ndvi(s)                     # (T, H, W), NaN wherever mask is False

print(f"cube {os.path.basename(s.path)}")
print(f"  values {s.values.shape} | mask {s.mask.shape} | ndvi {nd.shape} {nd.dtype}")
assert nd.shape == s.mask.shape
assert np.isnan(nd[~s.mask]).all(), "masked pixels leaked into NDVI"

with np.errstate(invalid="ignore"):
    ts_mean = np.nanmean(nd.reshape(nd.shape[0], -1), axis=1)
valid_frac = s.mask.reshape(nd.shape[0], -1).mean(axis=1)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(s.timestamps, ts_mean, "o-", ms=3, lw=0.7)
ax[0].set(title="spatial mean NDVI (masked)", ylim=(-0.2, 1.0))
ax[1].plot(s.timestamps, valid_frac, "o-", ms=3, lw=0.7, color="tab:red")
ax[1].set(title="clear-pixel fraction per timestep", ylim=(0, 1))
k = int(np.argmax(valid_frac))
im = ax[2].imshow(nd[k], vmin=-0.2, vmax=1.0, cmap="RdYlGn")
ax[2].set(title=f"NDVI @ {str(s.timestamps[k])[:10]} (clear {valid_frac[k]:.2f})")
plt.colorbar(im, ax=ax[2])
plt.tight_layout()
plt.show()

print(f"\nclearest timestep: min {np.nanmin(nd[k]):.3f} "
      f"median {np.nanmedian(nd[k]):.3f} max {np.nanmax(nd[k]):.3f}")
print("Alpine foreland summer vegetation 0.6 to 0.9, bare or urban 0.1 to 0.3, "
      "water negative")

## Phase 1.1 is done when

- [ ] Step 4: `48 passed`
- [ ] Step 7: 20 cubes load, all shape assertions pass, no overlapping pairs
- [ ] Step 7: `masked_fraction` printed per cube, median between 0.45 and 0.75
- [ ] Step 8: `exactly 5 d` well under 100%
- [ ] Step 9: a single seasonal green-up, not tracking the clear-fraction panel

Then record the cube checksums, not the cubes:

```
sha256sum data/raw/*.nc > data/raw.sha256
```

## Next

Phase 1.2: frozen encoder embeddings on these cubes. `.eval()` and
`torch.no_grad()`, no fine-tuning, ever.

Phase 1.3: `probes/cv.py`. Until that file exists, no number in this notebook is
a result. Everything above is a data-integrity check, which is what it is for.